In [1]:
import pandas as pd
from sentence_transformers import SentenceTransformer, util
import torch, json
from tqdm import tqdm

# ---- Paths ----
A_PATH = "/home/ubuntu/TW_MultiLabel_SMP/jupyter-notebooks/training_sets/train_20251012_043813.csv"  # queries
B_PATH = "/home/ubuntu/TW_MultiLabel_SMP/datasets/2000_Posts_Annotations - Combined_Dataset.csv"           # reference pool
A_EMB_PATH = "/home/ubuntu/embeddings/r8_embeddings_A.pt"
B_EMB_PATH = "/home/ubuntu/embeddings/r8_embeddings_B.pt"
SIM_JSON_OUT = "/home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_k3_500_r8.json"

# ---- Read ----
A = pd.read_csv(A_PATH)
B = pd.read_csv(B_PATH)

# ---- Build full_text (same style as your notebook) ----
A['full_text'] = A['title'].fillna('') + '. ' + A['selftext'].fillna('')
B['full_text'] = B.get('title','').fillna('') + '. ' + B.get('body','').fillna('')  # robust to missing cols

A.head(), B.head()


(        id    subreddit                                              title  \
 0   nljahp      assault  I've been sexually harassed and assaulted for ...   
 1   kbayek      assault               I would rather die than see an obgyn   
 2  1ldunz7  Miscarriage                                When does it start?   
 3  1er6d87     abortion  No fetal heartbeat because of Local reseller p...   
 4  1h30x8f     abortion                       I regret having an abortion.   
 
                                             selftext          created_utc  \
 0  I just made a post [here](https://www.reddit.c...  2021-05-26 15:08:59   
 1  I already stated I’m pregnant here before &amp...  2020-12-11 20:45:47   
 2  Just had my first prenatal visit- the ultrasou...  2025-06-17 18:20:43   
 3  Hi, I am a college student (21F) I live in the...  2024-08-13 12:16:58   
 4  This is my first post… I originally got Reddit...   2024-11-30 1:08:49   
 
                                                  url 

In [2]:
model = SentenceTransformer("all-mpnet-base-v2")

# Encode and save A
r6_emb_A = model.encode(
    A['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r6_emb_A, A_EMB_PATH)

# Encode and save B (do once; later you can just torch.load(B_EMB_PATH))
r6_emb_B = model.encode(
    B['full_text'].tolist(),
    convert_to_tensor=True,
    show_progress_bar=True,
    normalize_embeddings=True
)
torch.save(r6_emb_B, B_EMB_PATH)


Batches:   0%|          | 0/16 [00:00<?, ?it/s]

Batches:   0%|          | 0/63 [00:00<?, ?it/s]

In [3]:
emb_A = torch.load(A_EMB_PATH)
emb_B = torch.load(B_EMB_PATH)

In [4]:
k_max = 3

# Cosine similarity matrix: [len(A), len(B)]
# (normalize_embeddings=True above ⇒ cosine == dot)
sim_mat = util.cos_sim(r6_emb_A, r6_emb_B)  # torch tensor

# Top-k along B axis for each A row
top_vals, top_idx = torch.topk(sim_mat, k=k_max, dim=1)  # shapes: [len(A), k]

# Pack to dict: {a_row_index: [(b_index, score), ...]}
similar_posts = {}
for i in range(top_idx.size(0)):
    indices = top_idx[i].tolist()
    scores  = top_vals[i].tolist()
    similar_posts[i] = list(zip(indices, scores))

# Save JSON
with open(SIM_JSON_OUT, "w") as f:
    json.dump(similar_posts, f)

print(f"Saved cross-sim results to {SIM_JSON_OUT}")

Saved cross-sim results to /home/ubuntu/TW_MultiLabel_SMP/similarity_scores/cross_similar_posts_k3_500_r8.json
